# 🚗 CARLA VAE Training — Google Colab

Trains the **β-VAE encoder** on the 20 000 images collected from CARLA.

### Steps
1. Mount Google Drive
2. **Edit the two paths in Cell 2** to match your Drive folder
3. Run all cells in order
4. Download `vae_checkpoint.pth` → put it in your local `checkpoints/` folder

> **Your images are already on Drive as a folder — no zip needed. Just point `IMAGES_FOLDER` at it.**


In [ ]:
# ── Cell 1 · Mount Google Drive ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
# ── Cell 2 · Paths — EDIT THESE TWO LINES ─────────────────

# 👇 Full path to the images FOLDER you uploaded to Google Drive
#    Example: if you uploaded a folder called 'images' inside 'Carla_PPO'
#    your path would be: '/content/drive/MyDrive/Carla_PPO/images'
IMAGES_FOLDER = '/content/drive/MyDrive/Carla_PPO/images'

# 👇 Where checkpoints are saved (stays on Drive between Colab sessions)
CKPT_DIR = '/content/drive/MyDrive/Carla_PPO/checkpoints'

# ── Derived (do not edit) ──────────────────────────────────
import os
DATA_DIR  = IMAGES_FOLDER
VAE_CKPT  = f'{CKPT_DIR}/vae_checkpoint.pth'
DRIVE_BASE = os.path.dirname(CKPT_DIR)

os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Images folder → {DATA_DIR}')
print(f'Checkpoints   → {CKPT_DIR}')

In [ ]:
# ── Cell 3 · Verify images are accessible ─────────────────
import glob, os

pngs = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.png'), recursive=True))
if not pngs:
    raise FileNotFoundError(
        f'No PNG files found in: {DATA_DIR}\n'
        'Check that IMAGES_FOLDER in Cell 2 points to the correct Drive folder.'
    )
print(f'✅ Found {len(pngs)} images')
print(f'   First: {os.path.basename(pngs[0])}  |  Last: {os.path.basename(pngs[-1])}')

In [ ]:
# ── Cell 4 · Verify GPU ────────────────────────────────────
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 5 · Install dependencies ─────────────────────────
# torchvision and Pillow are already on Colab; safety check only
!pip install -q Pillow torchvision

In [ ]:
# ── Cell 6 · Hyperparameters ───────────────────────────────

# VAE
VAE_LATENT_DIM   = 64
VAE_BETA         = 1.0
VAE_BATCH_SIZE   = 128     # increase to 256 if you have a high-VRAM GPU
VAE_EPOCHS       = 50
VAE_LR           = 1e-3
VAE_WEIGHT_DECAY = 1e-5
VAE_VAL_SPLIT    = 0.1

# Image
IMG_WIDTH  = 64
IMG_HEIGHT = 64

print('Hyperparameters set ✅')

In [ ]:
# ── Cell 7 · VAE architecture (encoder.py inlined) ────────
import torch
import torch.nn as nn
import torch.nn.functional as F


class Encoder(nn.Module):
    def __init__(self, latent_dim=64, img_channels=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(img_channels, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64,  kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.flat_dim  = 128 * 8 * 8
        self.fc_mu     = nn.Linear(self.flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim, latent_dim)

    def forward(self, x):
        h = self.conv(x).view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder(nn.Module):
    def __init__(self, latent_dim=64, img_channels=3):
        super().__init__()
        self.flat_dim = 128 * 8 * 8
        self.fc       = nn.Linear(latent_dim, self.flat_dim)
        self.deconv   = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 32,  kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(32, img_channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        h = self.fc(z).view(z.size(0), 128, 8, 8)
        return self.deconv(h)


class VAE(nn.Module):
    def __init__(self, latent_dim=64, img_channels=3, beta=1.0):
        super().__init__()
        self.beta    = beta
        self.encoder = Encoder(latent_dim, img_channels)
        self.decoder = Decoder(latent_dim, img_channels)

    @staticmethod
    def reparameterise(mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.reparameterise(mu, log_var)
        return self.decoder(z), mu, log_var

    def encode(self, x):
        with torch.no_grad():
            mu, _ = self.encoder(x)
        return mu

    def loss(self, x, recon, mu, log_var):
        recon_loss = F.binary_cross_entropy(recon, x, reduction='sum') / x.size(0)
        kl_loss    = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
        return recon_loss + self.beta * kl_loss, recon_loss, kl_loss


print('VAE architecture defined ✅')

In [ ]:
# ── Cell 8 · Dataset & DataLoaders ────────────────────────
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms


class CarlaImageDataset(Dataset):
    def __init__(self, image_dir):
        self.paths = sorted(glob.glob(os.path.join(image_dir, '**', '*.png'), recursive=True))
        if not self.paths:
            raise FileNotFoundError(f'No PNG images found in {image_dir}')
        print(f'Dataset ready: {len(self.paths)} images')
        self.transform = transforms.Compose([
            transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return self.transform(Image.open(self.paths[idx]).convert('RGB'))


dataset    = CarlaImageDataset(DATA_DIR)
val_size   = max(1, int(len(dataset) * VAE_VAL_SPLIT))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=VAE_BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=VAE_BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {train_size}  |  Val: {val_size}  |  Batch: {VAE_BATCH_SIZE}')
print(f'Steps/epoch: {len(train_loader)}')

In [ ]:
# ── Cell 9 · Sanity-check: visualise 8 sample images ──────
import matplotlib.pyplot as plt

samples = next(iter(train_loader))[:8]
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(samples[i].permute(1, 2, 0).numpy())
    ax.axis('off')
plt.suptitle('Sample training images', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10 · Build model + optimiser (resume if checkpoint exists) ─
import torch.optim as optim

model = VAE(latent_dim=VAE_LATENT_DIM, beta=VAE_BETA).to(DEVICE)
opt   = optim.Adam(model.parameters(), lr=VAE_LR, weight_decay=VAE_WEIGHT_DECAY)
sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5, verbose=True)

start_epoch = 0
best_val    = float('inf')

if os.path.exists(VAE_CKPT):
    state = torch.load(VAE_CKPT, map_location=DEVICE)
    model.load_state_dict(state['state_dict'])
    start_epoch = state['epoch']
    best_val    = state['val_loss']
    print(f'📂 Resumed from epoch {start_epoch}  (val_loss={best_val:.4f})')
else:
    print('Starting fresh training.')

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
# ── Cell 11 · Training loop ────────────────────────────────
import time

history = {'train': [], 'val': [], 'recon': [], 'kl': []}

print(f'\n🚀 VAE training | epochs={VAE_EPOCHS} | latent_dim={VAE_LATENT_DIM} | device={DEVICE}\n')

for epoch in range(start_epoch + 1, VAE_EPOCHS + 1):
    t0 = time.time()

    # ── Train ──────────────────────────────────────────────
    model.train()
    t_loss = t_recon = t_kl = 0.0
    for batch in train_loader:
        x = batch.to(DEVICE)
        recon, mu, logvar = model(x)
        loss, recon_l, kl_l = model.loss(x, recon, mu, logvar)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        t_loss  += loss.item()
        t_recon += recon_l.item()
        t_kl    += kl_l.item()

    n = len(train_loader)

    # ── Validate ───────────────────────────────────────────
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            x = batch.to(DEVICE)
            recon, mu, logvar = model(x)
            loss, _, _ = model.loss(x, recon, mu, logvar)
            v_loss += loss.item()

    v_loss /= len(val_loader)
    sched.step(v_loss)

    elapsed = time.time() - t0
    print(f'Epoch [{epoch:3d}/{VAE_EPOCHS}]  '
          f'train={t_loss/n:.2f}  '
          f'recon={t_recon/n:.2f}  '
          f'kl={t_kl/n:.4f}  '
          f'val={v_loss:.2f}  '
          f'({elapsed:.1f}s)')

    history['train'].append(t_loss / n)
    history['val'].append(v_loss)
    history['recon'].append(t_recon / n)
    history['kl'].append(t_kl / n)

    # ── Save best checkpoint ───────────────────────────────
    if v_loss < best_val:
        best_val = v_loss
        torch.save({
            'epoch'     : epoch,
            'val_loss'  : best_val,
            'latent_dim': VAE_LATENT_DIM,
            'beta'      : VAE_BETA,
            'state_dict': model.state_dict(),
        }, VAE_CKPT)
        print(f'  ✅ New best → saved  (val={best_val:.4f})')

print(f'\n✅ Training complete.  Best val loss: {best_val:.4f}')
print(f'   Checkpoint: {VAE_CKPT}')

In [ ]:
# ── Cell 12 · Plot training curves ────────────────────────
import matplotlib.pyplot as plt

ep = range(1, len(history['train']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(ep, history['train'], label='Train')
axes[0].plot(ep, history['val'],   label='Val')
axes[0].set_title('Total Loss'); axes[0].legend()

axes[1].plot(ep, history['recon'])
axes[1].set_title('Reconstruction Loss')

axes[2].plot(ep, history['kl'])
axes[2].set_title('KL Divergence')

for ax in axes:
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/training_curves.png', dpi=150)
plt.show()
print('Plot saved to Drive.')

In [ ]:
# ── Cell 13 · Visual reconstructions ──────────────────────
model.eval()
test_imgs = next(iter(val_loader))[:8].to(DEVICE)
with torch.no_grad():
    recons, _, _ = model(test_imgs)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(test_imgs[i].cpu().permute(1,2,0).numpy())
    axes[0, i].axis('off')
    axes[0, i].set_title('Original' if i == 0 else '')

    axes[1, i].imshow(recons[i].cpu().permute(1,2,0).numpy())
    axes[1, i].axis('off')
    axes[1, i].set_title('Recon' if i == 0 else '')

plt.suptitle('Originals (top) vs Reconstructions (bottom)')
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/reconstructions.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 14 · Download checkpoint to your PC ───────────────
# Checkpoint is already on Drive at CKPT_DIR.
# This cell also triggers a direct browser download:

from google.colab import files
files.download(VAE_CKPT)
print(f'Downloading {VAE_CKPT}')
print('Place it in:  E:\\Carla_PPO_project_the.py\\checkpoints\\vae_checkpoint.pth')